# Real Tools and Structured Output: The Model Proposes, Your Code Verifies

You have probably already been burned by an agent that touched something real.

Maybe you asked an AI to research something live and it came back with a confident, well-written answer, and one of its "facts" was invented, or its source link did not say what it claimed. Or you asked for output in an exact shape, a table, a list, a format your spreadsheet could eat, and you got something *almost* right: close enough to look finished, wrong enough to be useless until you fixed it by hand.

Week 1's agent never put you in that position, and that was deliberate. Its world was a padded room: a calculator that could only do arithmetic and a search that only knew three facts. Nothing it did could touch your files, your network, or the real web. It was built that way so you could learn the loop without consequences.

**This week the padding comes off.** The agent gets real hands: a tool that fetches live web pages, a tool that runs shell commands, a tool that reads real files. Yours are already built (Days 1 and 2). And the moment tools are real, "the reply looked about right" stops being a standard anyone can accept. A reply that is *almost* a valid request can read the wrong file, run the wrong command, or hang forever on a broken address. So this week the model stops asking in prose, and your code stops hoping it read the request correctly.

**The one idea:** Week 1, the model asked in prose and your code hoped it had understood. This week the model asks in JSON, and your code *proves* the request is valid before it acts: a known tool, the right arguments, the right types, or the request is rejected with the exact problem named, and the model tries again. Tools are the agent's hands. Schemas are its contract.

*Week 2 of the agent sprint, as a lesson. Days 1 and 2 (the real tools) are built; this lesson is the concept behind the whole week, and Day 3 (structured output) is the stair you are standing on.*

**What Week 2 is for:** experiencing the reliability jump, the moment the agent stops being a toy you supervise and starts being a tool you can check. It is not about a perfect research assistant by Friday. If a validation error fires and the model corrects itself in your trace, the week is working, even if the answer is ordinary.

**How to read this notebook.** Same rhythm as Week 1, on purpose: definition first, then overview, then an example, then the mechanics. Every code cell has a lead-in (purpose, what to expect, why that output is your checkpoint) and its header comments are that lead-in compressed. The vocabulary from Week 1 carries over unchanged: prompt, parser, registry, Observation. Where Week 2 upgrades a Week 1 idea, the notebook says so out loud.

## The Whole Thing, Once, Slowly: A Toy Walkthrough

Same ritual as Week 1: the entire week in one small story, before any code. The question below is a Day 4-style research question, small enough to hold in your head:

> What files are in my sandbox, what does the README say Project Falcon is, and what does the project page say its status is?

Answering it takes three tool calls and one final answer. Watch what changed since Week 1.

**Your code wraps the question in a new contract.** The old contract asked for Thought / Action / Action Input lines. The new one asks for JSON, and only JSON. A shortened version of what gets sent:

```
You are a research agent. You answer questions by using tools, then cite your sources.
Reply with ONLY a JSON object, no other text. To use a tool:
{"tool": "<tool name>", "arguments": { ... }}
Available tools: run_command (arguments: command, a list), read_file (arguments: rel_path),
fetch_url (arguments: url, timeout).
When you have enough to answer, reply with ONLY:
{"answer": "<your answer>", "citations": ["<file or URL>", ...]}

Question: What files are in my sandbox, what does the README say Project Falcon is,
and what does the project page say its status is?
```

**The model replies, in JSON this time:**

```
{"tool": "run_command", "arguments": {"command": ["ls"]}}
```

**Your code checks before it acts. This is the new step.** Before anything runs, a validator (new this week) reads the JSON and checks it against the schema (also new): Is it valid JSON at all? Is `run_command` a real tool? Is `command` present, and is it a list? All yes. Only now does your code run the command and glue the result on as an Observation:

```
Observation: README.md  data.csv
```

**Turn 2.** Same dance, second tool:

```
{"tool": "read_file", "arguments": {"rel_path": "README.md"}}
```

Checked, run, and the file's text comes back as the Observation: Project Falcon is a tiny data logger for greenhouse sensors, recording temperature and humidity every minute, with a pointer to the project page for status.

**Turn 3. The model slips.** It asks for the project page, but writes the timeout the way a human would say it:

```
{"tool": "fetch_url", "arguments": {"url": "https://falcon.example.com/status", "timeout": "soon"}}
```

Week 1's parser would have shrugged and handed your code the string "soon" to deal with. The validator does not shrug. It rejects the call *before the tool runs* and says exactly why:

```
Observation: invalid tool call (bad_arguments): arguments.timeout must be a number, got a string. Reply with ONLY a JSON object: {"tool": ..., "arguments": {...}}.
```

That error is just text in the prompt, the same correction mechanism as Week 1's malformed-reply fix. The difference is how much the error knows: not "wrong format," but the exact field, the expected type, and what arrived instead.

**Turn 4. The model corrects itself:**

```
{"tool": "fetch_url", "arguments": {"url": "https://falcon.example.com/status", "timeout": 15}}
```

Checked, run, Observation: the status page says Project Falcon is in beta, logging runs nightly, dashboard not public yet.

**Turn 5. The final answer is structured too,** and it carries its receipts:

```
{"answer": "Project Falcon is a tiny data logger for greenhouse sensors (temperature and humidity, every minute). Its status page says it is in beta: logging runs nightly and the dashboard is not public yet.",
 "citations": ["README.md", "https://falcon.example.com/status"]}
```

Your code checks the answer's shape (an answer string, a list of citations), sees it is valid, and hands it to you. Three real tool calls, one rejected call that never ran, one answer you can verify against its citations. That is the whole week. Everything after this section is one piece of this story, taken slowly.

## Anatomy of a Validated Turn

Week 1's turn had four lines. Week 2's turn has a gate in the middle of it. Here is the same turn, both weeks:

| | Week 1 | Week 2 |
|---|---|---|
| The model asks with | Prose lines: `Action:` / `Action Input:` | One JSON object: `{"tool": ..., "arguments": {...}}` |
| Your code reads it with | A parser: two regexes and hope | A validator: a schema, checked field by field |
| A bad request is | Discovered late (or never) | Rejected *before* the tool runs, with the field named |
| The tools are | Toys in a padded room | Real: web, shell, filesystem |
| A wrong answer costs | A wrong sum | A wrong file read, a wrong command run |

The loop itself did not change. Ask, read the reply, act, append the Observation, repeat. What changed is the standard the reply must meet before your code acts on it, and the consequences riding on that standard. Keep the Week 1 picture in your head; this week hangs one new idea on it: **verification before action.**

### Setup: one library this week

**Purpose of the cell below:** load `json`, the one library this week's lesson is built on. Week 1 needed pattern-matching (`re`) because the model's requests were prose. This week's requests arrive as JSON, and Python has a built-in reader and writer for it.

**What to expect:** nothing printed, and that is correct. If a later cell complains about a missing name, this cell did not run.

In [1]:
# Setup: load json, the reader/writer for this week's request format.
# json.loads = text -> Python data. json.dumps = Python data -> text.
import json

## JSON: A Shape Machines Can Read Exactly

**Definition.** JSON is a way of writing data as plain text using names and values: `"url": "https://example.com"`, numbers without quotes, lists in square brackets, and boxes inside boxes in curly braces. It looks like a small, strict cousin of a Python dictionary, because it is one: the same shape, written as text so it can travel inside a prompt.

**Why it matters here.** Week 1's request was prose wearing a uniform:

```
Action: fetch_url
Action Input: https://falcon.example.com/status, soon
```

A human reads that fine. Code reads it with regexes, and regexes squint: where does the URL end and the timeout begin? Is "soon" a value or a comment? The Week 2 request leaves nothing to squint at:

```
{"tool": "fetch_url", "arguments": {"url": "https://falcon.example.com/status", "timeout": 15}}
```

Every piece has a name, every value has an exact type (`15` is a number; `"soon"` would be a string, and the difference is visible in the text itself, quotes or no quotes). Code does not interpret this. It *parses* it, exactly, or it fails loudly. That exactness is what the rest of the week is built on.

**The cell below** converts between the two worlds. It builds a tool call as Python data, writes it as JSON text (that is the text the model must produce), then reads the text back into data (that is what your code receives). *Expect:* the JSON text printed, then the name and the timeout pulled back out of the parsed data, with the timeout reporting its type as `int`, a number, not a string. **Why this output is your checkpoint:** the round trip is the whole mechanic of the week in miniature. Text the model writes becomes data your code can check, field by field. If the round trip works, everything after it is bookkeeping.

In [2]:
# JSON round trip: data -> text (what the model writes) -> data (what your code checks).
# Build the call as Python data first, the way your code wishes the world worked.
call = {"tool": "fetch_url",
        "arguments": {"url": "https://falcon.example.com/status", "timeout": 15}}

text = json.dumps(call)          # data -> text: this string is what a model reply should be
print("As text:    ", text)

back = json.loads(text)          # text -> data: what your code actually receives
print("Tool name:  ", back["tool"])
print("Timeout:    ", back["arguments"]["timeout"], type(back["arguments"]["timeout"]).__name__)

As text:     {"tool": "fetch_url", "arguments": {"url": "https://falcon.example.com/status", "timeout": 15}}
Tool name:   fetch_url
Timeout:     15 int


## The Schema: The Contract, Written as Data

**Definition.** A *schema* is a written description of what shape is allowed. For this week it answers three questions about every tool call: Which tools exist? What arguments does each one take? For each argument, what type must it be, and is it required or optional? A schema is not code that runs. It is a reference table the validator checks requests against, the way a bouncer checks IDs against a list.

**Your schema covers your three real tools,** using their real signatures from Days 1 and 2:

- `fetch_url(url, timeout=15)` — `url`: a string, required. `timeout`: a number, optional.
- `run_command(command, timeout=30)` — `command`: a list (your no-shell rule: the command arrives as a list of words, never one string to be interpreted), required. `timeout`: a number, optional.
- `read_file(rel_path)` — `rel_path`: a string, required.

Each argument is stored as a pair: `(type, required?)`. One small wrinkle worth knowing before you trip on it: JSON has a single number type, but Python splits numbers into `int` and `float`, and `json.loads` hands you an `int` for `15` and a `float` for `15.5`. So a "number" field must accept either, and the checker in the next section handles that. (In Python, `True` is technically an `int` too. A timeout of `true` is not a number in any sense that matters, and the checker refuses it.)

**The cell below** defines `TOOL_SCHEMAS` and prints it back as a readable table. *Expect:* three tools, each with its arguments, types, and required/optional marks, matching the list above. **Why this output is your checkpoint:** the schema *is* the week's contract, and this printout is the contract in plain sight. If a tool or type is wrong here, every verdict in the rest of the notebook is wrong with it.

In [3]:
# TOOL_SCHEMAS: the contract, as data. tool name -> {argument: (type, required?)}.
# Written from the real Day 1/2 signatures. The validator in the next section
# checks every model request against this table before any tool runs.
TOOL_SCHEMAS = {
    "fetch_url":   {"url": (str, True),  "timeout": (float, False)},
    "run_command": {"command": (list, True), "timeout": (float, False)},
    "read_file":   {"rel_path": (str, True)},
}

TYPE_NAMES = {str: "string", float: "number", list: "list"}

for tool, args in TOOL_SCHEMAS.items():
    for arg, (typ, required) in args.items():
        print(f"{tool:12} {arg:10} {TYPE_NAMES[typ]:7} {'required' if required else 'optional'}")

fetch_url    url        string  required
fetch_url    timeout    number  optional
run_command  command    list    required
run_command  timeout    number  optional
read_file    rel_path   string  required


## validate_tool_call: Trust, but Verify (Again)

**Definition.** `validate_tool_call` is this week's equivalent of Week 1's parser, grown up. It takes the model's raw reply text and returns a verdict as a small dictionary: `{"ok": True, "tool": ..., "arguments": ...}` when the call may run, or `{"ok": False, "error": ..., "detail": ...}` when it may not, with the detail naming the exact problem. Notice the shape of that failure: it is the same ok/error/detail shape your Day 1 and Day 2 tools already return. That is not an accident. Failures in this system all speak the same language, so the loop can handle any of them the same way.

**The walkthrough.** The validator runs four gates, in order. Watch it work on the toy walkthrough's slip, the `fetch_url` call with `"timeout": "soon"`:

1. **It sees this:** the raw reply text. **It runs:** `json.loads`, the JSON reader. **This means:** if the text is not valid JSON at all (the classic case: the model wrapped the JSON in prose, "Here is the tool call: {...}"), the verdict is `invalid_json`, and nothing else is even attempted.
2. **It sees this:** a parsed object. **It runs:** a shape check: does it have `tool` and `arguments`? **This means:** a JSON object of the wrong shape is rejected here, still before anything runs.
3. **It sees this:** the tool name. **It runs:** a lookup in `TOOL_SCHEMAS`. **This means:** a made-up tool (`google_it`) gets `unknown_tool`, and the detail lists what *does* exist.
4. **It sees this:** the arguments. **It runs:** a field-by-field check against the schema: required ones present, types matching. **This means:** `"soon"` fails here, and the verdict is `bad_arguments` with the detail `arguments.timeout must be a number, got a string`. The exact field. The expected type. What arrived instead. That one sentence is what the model will be told, and it is specific enough for the model to fix the call in one try.

All four gates pass, and only then, does the verdict say ok.

<div align="center">

A raw model reply

`{"tool": "fetch_url", "arguments": {"url": "...", "timeout": "soon"}}`

↓

Gate 1: does it parse as JSON?

yes

↓

Gate 2: does it have `tool` and `arguments`?

yes

↓

Gate 3: is the tool in the schema?

`fetch_url` ✓

↓

Gate 4: does every argument match its type?

`timeout` is `"soon"`, a string

↓

Rejected, before the tool ever runs

`{"ok": False, "error": "bad_arguments", "detail": "arguments.timeout must be a number, got a string"}`

</div>

**The cell below** defines the validator and runs four test replies through it. The lead-in for each, so you know what to expect:

1. A prose-wrapped call ("Here is the tool call: {...}") → `invalid_json`. The most common real-world failure, per your Day 3 sheet.
2. A clean-JSON call to a tool that does not exist (`google_it`) → `unknown_tool`, with the real tools listed.
3. The walkthrough's slip, `timeout: "soon"` → `bad_arguments`, field named.
4. A fully valid `read_file` call → `ok: True`, with the tool and arguments handed back, ready to run.

**Why this output is your checkpoint:** three loud, specific rejections and one clean pass is the entire Day 3 done-criteria in embryo: invalid calls die here, named, before your code acts. Everything later in the notebook trusts this function.

In [4]:
# validate_tool_call: the four gates, in code.
# Takes raw reply text; returns {"ok": True, "tool", "arguments"} or
# {"ok": False, "error", "detail"} -- the same ok/error/detail shape real
# tools return, so the loop handles every failure in the system the same way.
def _type_matches(value, expected):
    if expected is float:   # a JSON number: accept int or float, never True/False
        return isinstance(value, (int, float)) and not isinstance(value, bool)
    return isinstance(value, expected)

def _type_name(value):
    if isinstance(value, bool):
        return "boolean"
    if isinstance(value, str):
        return "string"
    if isinstance(value, (int, float)):
        return "number"
    if isinstance(value, list):
        return "list"
    if isinstance(value, dict):
        return "object"
    return type(value).__name__

def validate_tool_call(raw_text):
    """Check a raw model reply against TOOL_SCHEMAS. A verdict, never an exception."""
    # Gate 1: is it JSON at all? (Prose-wrapped JSON dies here, on purpose.)
    try:
        call = json.loads(raw_text)
    except (json.JSONDecodeError, TypeError):
        return {"ok": False, "error": "invalid_json",
                "detail": "reply was not valid JSON"}
    # Gate 2: is it an object with the two keys a call must have?
    if not isinstance(call, dict) or "tool" not in call or "arguments" not in call:
        return {"ok": False, "error": "invalid_json",
                "detail": "reply must be a JSON object with 'tool' and 'arguments'"}
    name, args = call["tool"], call["arguments"]
    # Gate 3: is the tool real?
    if name not in TOOL_SCHEMAS:
        return {"ok": False, "error": "unknown_tool",
                "detail": f"unknown tool '{name}'. Available: {', '.join(TOOL_SCHEMAS)}"}
    if not isinstance(args, dict):
        return {"ok": False, "error": "bad_arguments",
                "detail": "'arguments' must be an object"}
    # Gate 4: every argument, field by field: present if required, right type.
    for arg, (expected, required) in TOOL_SCHEMAS[name].items():
        if arg not in args:
            if required:
                return {"ok": False, "error": "bad_arguments",
                        "detail": f"arguments.{arg} is required"}
            continue
        if not _type_matches(args[arg], expected):
            return {"ok": False, "error": "bad_arguments",
                    "detail": f"arguments.{arg} must be a {TYPE_NAMES[expected]}, got a {_type_name(args[arg])}"}
    return {"ok": True, "tool": name, "arguments": args}

tests = [
    'Here is the tool call: {"tool": "read_file", "arguments": {"rel_path": "README.md"}}',
    '{"tool": "google_it", "arguments": {"query": "Project Falcon"}}',
    '{"tool": "fetch_url", "arguments": {"url": "https://falcon.example.com/status", "timeout": "soon"}}',
    '{"tool": "read_file", "arguments": {"rel_path": "README.md"}}',
]
for t in tests:
    print(validate_tool_call(t))

{'ok': False, 'error': 'invalid_json', 'detail': 'reply was not valid JSON'}
{'ok': False, 'error': 'unknown_tool', 'detail': "unknown tool 'google_it'. Available: fetch_url, run_command, read_file"}
{'ok': False, 'error': 'bad_arguments', 'detail': 'arguments.timeout must be a number, got a string'}
{'ok': True, 'tool': 'read_file', 'arguments': {'rel_path': 'README.md'}}


## The Retry: The Error Goes Back as Text

**Definition.** The *retry* is what happens after a rejection: the verdict's error is written into an Observation, appended to the prompt, and the model gets another turn. No new machinery. It is Week 1's malformed-reply correction, word for word in structure, with one upgrade: the message the model receives.

Compare the two corrections:

- **Week 1:** "I couldn't parse an action from your last response. Reply using exactly the format..." True, and vague. The model knows *that* it failed, not *where*.
- **Week 2:** "invalid tool call (bad_arguments): arguments.timeout must be a number, got a string." The model knows the field, the expectation, and its own mistake, because the validator's detail was written to be pasted.

That specificity is the entire reliability story. A vague error makes the model guess at a fix; a named error makes the fix obvious. When you watch the retry work in the cell below, watch *why* it works: not because the model got smarter between turns, but because the feedback got sharper.

**The cell below** runs the retry in miniature: a scripted model first sends the prose-wrapped call (the `invalid_json` case), receives the error Observation, then sends the corrected call. *Expect:* the first verdict (`invalid_json`), the exact Observation text that would be glued onto the prompt, then the second verdict (`ok: True`). **Why this output is your checkpoint:** the loop section next does this automatically, inside the full run. Here you see the moving part alone, so when it fires inside the big demo, you recognize it.

In [5]:
# The retry, in miniature: one bad reply, the error Observation, one corrected reply.
# The "model" here is a two-item script; the validate-and-report steps are real.
def error_observation(verdict):
    """Turn a failed verdict into the Observation text the model will read."""
    return (f"Observation: invalid tool call ({verdict['error']}): {verdict['detail']}. "
            f"Reply with ONLY a JSON object: {{\"tool\": ..., \"arguments\": {{...}}}}.")

scripted_replies = [
    'Here is the tool call: {"tool": "read_file", "arguments": {"rel_path": "README.md"}}',
    '{"tool": "read_file", "arguments": {"rel_path": "README.md"}}',
]

for reply in scripted_replies:
    verdict = validate_tool_call(reply)
    print("Reply:  ", reply[:60], "..." if len(reply) > 60 else "")
    print("Verdict:", verdict)
    if not verdict["ok"]:
        print("Fed back:", error_observation(verdict))
    print()

Reply:   Here is the tool call: {"tool": "read_file", "arguments": {" ...
Verdict: {'ok': False, 'error': 'invalid_json', 'detail': 'reply was not valid JSON'}
Fed back: Observation: invalid tool call (invalid_json): reply was not valid JSON. Reply with ONLY a JSON object: {"tool": ..., "arguments": {...}}.

Reply:   {"tool": "read_file", "arguments": {"rel_path": "README.md"} ...
Verdict: {'ok': True, 'tool': 'read_file', 'arguments': {'rel_path': 'README.md'}}



## Real Tools, Real Consequences

Structured output only matters because of what sits behind it. Quick grounding in your three real tools, the ones from Days 1 and 2, because the validator exists to protect exactly these:

- **`fetch_url(url, timeout=15)`** goes to the live web and returns a page's text. Its failure modes are weather, not bugs: timeouts, 404s, pages that are not HTML at all. Your Day 1 build returns those as structured error dictionaries (`{"ok": False, "error": "http_404", ...}`), never as crashes. A slow or dead URL is a fact the model gets told, not an exception that kills the run.
- **`run_command(command, timeout=30)`** runs a shell command, the most dangerous tool an agent can own. Your Day 2 build fences it three ways: the command must arrive as a *list* (no shell string to smuggle extra commands into), it runs only inside one sandbox directory, and a timeout kills it. Output is capped so a runaway command cannot flood the prompt.
- **`read_file(rel_path)`** reads a file, but only inside the same sandbox: the path is resolved and checked, and anything that escapes (`../` tricks) is rejected with a `path_escape` error. The agent can read what you put in its room. It cannot wander the house.

Now the validator's job should be obvious. Week 1's tools were harmless, so a sloppy request cost nothing. These tools read real files, run real commands, and wait on a real network. **The schema check is the checkpoint that happens while the request is still just words.** You have a name for this in your own process language: it is a YMC gate, "you must check," applied to machine output, every turn, before the hands move. Week 1 taught the model to ask. Week 2 teaches your code to check the ask before honoring it.

## The Loop, Upgraded

**Definition.** The Week 2 *loop* is Week 1's loop with three parts swapped, and nothing else changed:

1. **The contract** now asks for JSON only (tool calls, and a structured final answer with citations).
2. **The parse step is now the validate step.** A reply is checked by `validate_tool_call`; a failed verdict becomes an error Observation and the turn repeats, a passing one is dispatched with its arguments.
3. **Dispatch takes arguments, not one string.** Week 1's `run_tool(name, arg)` becomes, in effect, "run this tool with these named arguments," which is what lets one registry serve tools with different shapes of input.

The mock tools in the cell below stand in for your real ones: a two-file sandbox and one canned web page, returning the same ok/error dictionaries your real tools return. The mock *model* replays the toy walkthrough's script, including the `timeout: "soon"` slip.

**When you run the cell,** expect the turns in order, each printing the model's reply and what your code did with it: validated and ran (turns 1 and 2), rejected with the named error (turn 3, and watch the full updated prompt print once there: the error Observation is visible at the bottom of it, which is the retry mechanism caught in the act), corrected and ran (turn 4), and the final answer with its citations, shape-checked and accepted (turn 5). **Why this output is your checkpoint:** it is the whole week in one run: 3+ tool calls, one rejection that never reached a tool, one self-correction, one cited answer. Those are the Day 4 and Day 5 done-criteria, rehearsed against the mock before you wire the real model in.

> **Simulated model ahead.** The next cell uses `MockModel`, the same stand-in as Week 1: pre-written replies, popped in order. The tools are simulated too (a canned sandbox and one canned page), returning the same ok/error dictionary shapes your real tools return. Everything the loop itself does (validate, reject, feed back, dispatch, check the final answer) is the real logic. Only the sources of text are fake, which is what makes the demo reproducible.

In [6]:
# MockModel + the upgraded loop: validate first, then dispatch, errors fed back.
# The mock model replays the toy walkthrough, slip included. The mock tools
# return the same {"ok": ...} dictionaries the real Day 1/2 tools return.
# Note the prompt is slotted with .replace, not .format: the template is full
# of literal JSON braces, and .format would read them as its own placeholders.
class MockModel:
    """Deterministic stand-in for ask(): pops canned replies in order.
    The loop cannot tell the difference -- it only ever sees text."""

    def __init__(self, script):
        self.script = list(script)
        self.calls = 0

    def __call__(self, prompt):
        self.calls += 1
        if not self.script:
            return '{"answer": "I am out of scripted replies.", "citations": []}'
        return self.script.pop(0)

# --- mock tools: a two-file sandbox and one canned page ---------------------
SANDBOX_FILES = {
    "README.md": "Project Falcon is a tiny data logger for greenhouse sensors. "
                 "It records temperature and humidity every minute. "
                 "See the project page for status.",
}
WEB_PAGES = {
    "https://falcon.example.com/status":
        "Project Falcon status: beta. Logging runs nightly; the dashboard is not public yet.",
}

def mock_run_command(command, timeout=30):
    if command == ["ls"]:
        return {"ok": True, "stdout": "README.md  data.csv"}
    return {"ok": False, "error": "bad_command", "detail": "mock sandbox only knows ['ls']"}

def mock_read_file(rel_path):
    if rel_path in SANDBOX_FILES:
        return {"ok": True, "path": rel_path, "text": SANDBOX_FILES[rel_path]}
    return {"ok": False, "error": "not_found", "requested": rel_path}

def mock_fetch_url(url, timeout=15):
    if url in WEB_PAGES:
        return {"ok": True, "url": url, "text": WEB_PAGES[url]}
    return {"ok": False, "error": "http_404", "detail": f"Server returned 404 for {url}"}

MOCK_TOOLS = {"run_command": mock_run_command, "read_file": mock_read_file,
              "fetch_url": mock_fetch_url}

def observation_from(result):
    """Wrap a tool result dict as the Observation text the model reads."""
    if result["ok"]:
        body = result.get("text") or result.get("stdout") or ""
        return f"Observation: {body}"
    return f"Observation: tool error ({result['error']}): {result.get('detail', '')}"

WEEK2_PROMPT = """You are a research agent. You answer questions by using tools, then cite your sources.
Reply with ONLY a JSON object, no other text. To use a tool:
{"tool": "<tool name>", "arguments": { ... }}
Available tools: run_command (arguments: command, a list), read_file (arguments: rel_path),
fetch_url (arguments: url, timeout).
When you have enough to answer, reply with ONLY:
{"answer": "<your answer>", "citations": ["<file or URL>", ...]}

Question: {question}
"""

def run(question, model, max_iterations=10):
    """The Week 2 loop: ask, validate, dispatch, append, repeat.
    Tool calls are schema-checked before anything runs; the final answer
    must be a JSON object with an answer string and a citations list."""
    prompt = WEEK2_PROMPT.replace("{question}", question)  # not .format: JSON braces are literal
    for turn in range(1, max_iterations + 1):
        reply = model(prompt)
        print(f"--- Turn {turn}: model reply ---")
        print(reply)
        # Final answer attempt? It is JSON with an "answer" key, no "tool" key.
        try:
            parsed = json.loads(reply)
        except json.JSONDecodeError:
            parsed = None
        if isinstance(parsed, dict) and "answer" in parsed:
            if isinstance(parsed["answer"], str) and isinstance(parsed.get("citations"), list):
                print("Answer accepted: shape valid, citations present.\n")
                return parsed
            obs = ("Observation: invalid final answer: it must be a JSON object with "
                   "an 'answer' string and a 'citations' list.")
        else:
            verdict = validate_tool_call(reply)
            if not verdict["ok"]:
                obs = error_observation(verdict)
                print("Verdict: REJECTED ->", verdict["error"], "-", verdict["detail"])
                print("Fed back:", obs)
                prompt += "\n" + reply + "\n" + obs + "\n"
                print("\n===== Updated prompt after the rejection =====")
                print(prompt)
                print("===== end of prompt =====\n")
                continue
            print("Verdict: ok ->", verdict["tool"], verdict["arguments"])
            result = MOCK_TOOLS[verdict["tool"]](**verdict["arguments"])
            obs = observation_from(result)
        print(obs, "\n")
        prompt += "\n" + reply + "\n" + obs + "\n"
    return {"answer": "max iterations reached", "citations": []}

script = [
    '{"tool": "run_command", "arguments": {"command": ["ls"]}}',
    '{"tool": "read_file", "arguments": {"rel_path": "README.md"}}',
    '{"tool": "fetch_url", "arguments": {"url": "https://falcon.example.com/status", "timeout": "soon"}}',
    '{"tool": "fetch_url", "arguments": {"url": "https://falcon.example.com/status", "timeout": 15}}',
    '{"answer": "Project Falcon is a tiny data logger for greenhouse sensors (temperature and humidity, every minute). Its status page says it is in beta: logging runs nightly and the dashboard is not public yet.", "citations": ["README.md", "https://falcon.example.com/status"]}',
]

result = run("What files are in my sandbox, what does the README say Project Falcon is, "
             "and what does the project page say its status is?", MockModel(script))
print("FINAL ANSWER:", result["answer"])
print("CITATIONS:   ", result["citations"])

--- Turn 1: model reply ---
{"tool": "run_command", "arguments": {"command": ["ls"]}}
Verdict: ok -> run_command {'command': ['ls']}
Observation: README.md  data.csv 

--- Turn 2: model reply ---
{"tool": "read_file", "arguments": {"rel_path": "README.md"}}
Verdict: ok -> read_file {'rel_path': 'README.md'}
Observation: Project Falcon is a tiny data logger for greenhouse sensors. It records temperature and humidity every minute. See the project page for status. 

--- Turn 3: model reply ---
{"tool": "fetch_url", "arguments": {"url": "https://falcon.example.com/status", "timeout": "soon"}}
Verdict: REJECTED -> bad_arguments - arguments.timeout must be a number, got a string
Fed back: Observation: invalid tool call (bad_arguments): arguments.timeout must be a number, got a string. Reply with ONLY a JSON object: {"tool": ..., "arguments": {...}}.

===== Updated prompt after the rejection =====
You are a research agent. You answer questions by using tools, then cite your sources.
Reply wit

## Going Live: Your Build

The demo above ran on a simulated model and simulated tools. Your build runs the same loop against your real files and your real model call. What changes:

- `MockModel` becomes your `ask_gateway(prompt)`: text in, text out, the Week 1 swap. The loop cannot tell the difference.
- The mock tools become your Day 1 and Day 2 tools, which return the same ok/error dictionary shapes the mocks returned. That shape-matching was deliberate: the lesson could be rehearsed end to end before a single real command ran.

### Where each piece lives in your repo

| Notebook section | Your file | Day |
|---|---|---|
| `fetch_url` | `week-02-tools-structured-output/web_fetch.py` | Day 1 |
| `run_command`, `read_file` | `week-02-tools-structured-output/shell_tools.py` | Day 2 |
| `TOOL_SCHEMAS`, `validate_tool_call` | `week-02-tools-structured-output/structured.py` | Day 3 |
| The upgraded loop | your loop, wired to the real registry | Day 4 |
| The three breakages | `BREAKAGE_LOG.md` | Day 5 |

(`tools.py` holds the registry itself, `TOOLS` and the dispatch, carried over from Week 1's pattern.)

### Spot the difference (exercise)

Three things differ between the notebook's demo and your real build. Narrate each one out loud:

1. **Dispatch.** The notebook calls `MOCK_TOOLS[name](**arguments)`: the arguments dictionary unpacked into named parameters. Your Week 2 `run_tool(name, arg)` still passes *one string*. What has to change in `tools.py` when arguments become structured, and what breaks if it does not?
2. **The error language.** Week 1's parser returned `None` and the loop invented a generic correction. The validator returns a named error with a detail written to be pasted into the prompt. Why does the detail's exact wording matter more than the error's category?
3. **The tools were already speaking this language.** Your Day 1 and Day 2 tools return `{"ok": False, "error": ..., "detail": ...}` dictionaries. The validator returns the same shape. What does the loop gain from every failure in the system arriving in one shape?

## Your Build, Day by Day

| Day | You build | In file | Done when |
|---|---|---|---|
| 1 | `fetch_url`: live web pages as text, unhappy paths as structured errors | `web_fetch.py` | A page fetch prints clean text; a 404, a timeout, and a non-HTML response each come back as named errors, not crashes. **Shipped Oct 14.** |
| 2 | `run_command` + `read_file`, sandboxed | `shell_tools.py` | All three tools run through the registry; a path escaping the sandbox is rejected; a command that runs long is killed by the timeout. **Shipped Oct 16.** |
| 3 | The schema + validator: tool calls as JSON, checked before anything runs | `structured.py` | An invalid call is rejected with the specific field named, and the model corrects itself on the next turn, visible in the trace |
| 4 | The research assistant: the upgraded loop, wired end to end | your loop + `tools.py` | A real, previously-unknown question is answered with 3+ tool calls, and every factual claim cites a source you verified |
| 5 | Prove it: break it on purpose, log it | `BREAKAGE_LOG.md` | The log covers a mid-run tool failure, a timeout, and a double validation failure, and the agent recovered from at least one without dying |

**Day 3, the stair you are on.** The day sheet's route: define the schemas for your three tools as data (the notebook's Schema section is the shape), write `validate_tool_call` to your skeleton's contract (verdict dictionaries: `{"ok": True, ...}` or `{"ok": False, "error", "detail"}`), swap the loop's parse step for the validate step, then break it three ways on purpose (malformed JSON, wrong tool name, wrong-typed argument) and watch all three come back named. One decision the sheet forces: the schemas are hand-checked data in your skeleton, no library required; pydantic is the named alternative if you ever outgrow the hand-rolled version. Pick once, today, and move.

**Day 4** is assembly, not invention: the loop from this lesson, your real registry, your real model. The new discipline is citations: an answer claim with no source behind it does not ship.

**Day 5** is the Week 1 ritual with real stakes: a URL that 404s mid-run, a command that times out, a reply that fails validation twice in a row. Recover from at least one, log all three.

## Reference Code: The Build, File by File

The Week 2 build as files, the way it lives (or will live) in your repo. Reference, not a copy source: read for shape, type your own. Days 1 and 2 below are your real code with the comments expanded; Day 3 is this lesson's validator assembled into the file your skeleton describes; Day 4 is the upgraded loop wired to the real imports.

A naming bridge, so nothing jars: the notebook's mock tools return the same dictionaries as your real tools, and the validator's verdicts use that same shape too. In this build, *everything* reports as `{"ok": ...}`. One language for success and failure, all the way down.

### Reference: `web_fetch.py` (Day 1)

**Walkthrough.** `fetch_url(url, timeout=15)` is the agent's window on the live web: give it an address, get back the page's visible text. Two ideas carry the file. First, the unhappy paths are the design, not an afterthought: timeout, 404, other bad statuses, and non-HTML responses each return a named error dictionary, because a research agent that dies on the first dead link is useless. Second, a small parser class (`_TextExtractor`) strips the HTML down to visible text, skipping scripts and styles, and a character cap keeps one huge page from flooding the prompt. In the loop, this file is called only through the registry, and its return value, ok or error, becomes the Observation either way.

```python
"""Fetch a URL's text. Unhappy paths return structured errors, never exceptions."""

import httpx
from html.parser import HTMLParser

MAX_CHARS = 50000  # cap: the model's context is not infinite


class _TextExtractor(HTMLParser):
    """Collects visible text, skipping script and style content.
    A web page is mostly markup the model should never see; this class
    keeps the words and drops the scaffolding."""

    def __init__(self):
        super().__init__()
        self.parts = []
        self._skip = 0  # depth inside a script/style tag

    def handle_starttag(self, tag, attrs):
        if tag in ("script", "style"):
            self._skip += 1          # entering a skip zone: stop collecting

    def handle_endtag(self, tag):
        if tag in ("script", "style"):
            self._skip = max(0, self._skip - 1)   # leaving: resume

    def handle_data(self, data):
        if self._skip == 0:
            self.parts.append(data)  # visible text only

    def text(self):
        raw = " ".join(self.parts)
        return " ".join(raw.split())  # collapse all whitespace runs


def fetch_url(url, timeout=15):
    """Return {'ok': True, 'url': ..., 'text': ...} or
    {'ok': False, 'error': ..., 'detail': ...}. Never raises:
    every failure is a fact the model gets told, in the same shape."""
    # The request itself can fail two ways: too slow, or not at all.
    try:
        response = httpx.get(url, timeout=timeout, follow_redirects=True)
    except httpx.TimeoutException as e:
        return {"ok": False, "error": "timeout", "detail": str(e)}
    except httpx.RequestError as e:
        return {"ok": False, "error": "request_error", "detail": str(e)}

    # The server answered, but with a failure status. 404 gets its own
    # name because it is the one a research agent meets most.
    if response.status_code == 404:
        return {"ok": False, "error": "http_404", "detail": f"Server returned 404 for {url}"}
    if not 200 <= response.status_code < 300:
        return {"ok": False, "error": "http_error", "detail": f"Server returned {response.status_code} for {url}"}

    # A 200 that is not HTML (a PDF, an image) is not readable text.
    content_type = response.headers.get("content-type", "")
    if "text/html" not in content_type:
        return {"ok": False, "error": "non_html", "detail": f"Content-Type is {content_type!r}, not HTML"}

    # Happy path: strip to visible text, cap the length, hand it back.
    extractor = _TextExtractor()
    extractor.feed(response.text)
    text = extractor.text()[:MAX_CHARS]
    return {"ok": True, "url": url, "text": text}
```

### Reference: `shell_tools.py` (Day 2)

**Walkthrough.** Two tools, one sandbox. `run_command(command, timeout=30)` runs a shell command, the most dangerous capability in the build, fenced three ways: the command must be a *list* of words (there is no shell to interpret a string, so nothing can be smuggled after a semicolon), it runs only inside `SAFE_DIR`, and the timeout kills it if it runs long. `read_file(rel_path)` reads a file from that same sandbox: the requested path is resolved to its real, absolute form and checked with `is_relative_to`, so `../` escapes are rejected before a single byte is read. Both return the ok/error dictionary shape, and both cap their output, because a tool that can flood the prompt can break the agent all by itself.

```python
"""Shell and filesystem tools, fenced into one sandbox directory."""

import subprocess
from pathlib import Path

# The only directory the agent is allowed to work in (read or run).
SAFE_DIR = r'D:\agent-sprint\week-02-tools-structured-output\sandbox'

# How many seconds an action gets before it is killed.
MAX_TIMEOUT = 30

# Cap on data coming back from a tool call: prevents flooding the context.
MAX_OUTPUT_BYTES = 20000

# The sandbox as a resolved Path, computed once: every check compares
# against this canonical form, so tricks with .. or symlinks cannot slip by.
_SAFE_DIR = Path(SAFE_DIR).resolve()


def run_command(command, timeout=MAX_TIMEOUT):
    """Run *command* (a list of words) inside the sandbox.
    Returns {'ok': True, 'stdout', 'stderr', 'exit_code', ...} or a named
    error dictionary. Never raises."""
    # 1. The no-shell rule: only a non-empty list is accepted. A list is
    #    handed to the OS as separate words; there is no shell string for
    #    extra commands to hide in.
    if not isinstance(command, list) or len(command) == 0:
        return {"ok": False, "error": "bad_command",
                "detail": "command must be a non empty list, e.g. ['python', '--version']"}

    # 2. Run it in SAFE_DIR, output captured. Killed if it exceeds timeout.
    try:
        result = subprocess.run(command, capture_output=True, text=True,
                                timeout=timeout, cwd=SAFE_DIR)
    except subprocess.TimeoutExpired:
        return {"ok": False, "error": "timeout",
                "detail": f"command timed out after {timeout} seconds"}
    except FileNotFoundError:
        return {"ok": False, "error": "not_found",
                "detail": f"program not found: {command[0]}"}

    # 3. Truncate the output so a noisy command cannot flood the prompt.
    stdout, stderr, truncated = result.stdout, result.stderr, False
    if len(stdout) > MAX_OUTPUT_BYTES:
        stdout, truncated = stdout[:MAX_OUTPUT_BYTES], True
    if len(stderr) > MAX_OUTPUT_BYTES:
        stderr, truncated = stderr[:MAX_OUTPUT_BYTES], True

    # 4. Structured output, the same shape as every tool in this build.
    return {"ok": True, "command": command, "stdout": stdout, "stderr": stderr,
            "exit_code": result.returncode, "truncated": truncated}


def read_file(rel_path):
    """Read a file inside the sandbox. Escapes are rejected before any
    read happens. Returns the ok/error dictionary shape, never raises."""
    # Resolve the requested path against the sandbox into its real form...
    target = (_SAFE_DIR / rel_path).resolve()
    # ...and reject anything that lands outside it. This is the check
    # that makes ../ harmless: the escape is computed, seen, and refused.
    if not target.is_relative_to(_SAFE_DIR):
        return {"ok": False, "error": "path_escape",
                "requested": rel_path, "resolved": str(target)}

    # The read itself can fail three ways: the file is not there, the
    # bytes are not text, or the OS refuses. Each gets its handling.
    try:
        text = target.read_text(encoding="utf-8")
    except FileNotFoundError:
        return {"ok": False, "error": "not_found", "requested": rel_path}
    except (UnicodeDecodeError, OSError):
        return {"ok": False, "error": "decode_error", "requested": rel_path}

    # Truncate giant files so one huge read cannot flood the context.
    truncated = False
    if len(text) > MAX_OUTPUT_BYTES:
        text, truncated = text[:MAX_OUTPUT_BYTES], True

    return {"ok": True, "path": str(target), "text": text, "truncated": truncated}
```

### Reference: `structured.py` (Day 3)

**Walkthrough.** The new file, assembled from this lesson's Schema and Validator sections to the contract in your Day 3 skeleton. `TOOL_SCHEMAS` is the contract as data. `validate_tool_call(raw_text)` runs the four gates and returns the verdict dictionary. `error_observation(verdict)` turns a rejection into the exact text the loop appends to the prompt, because the detail was written to be read by the model, not just logged for you. Keep this file free of imports from the rest of the build: in Week 6, these schemas become the eval harness's ground truth, and a file that stands alone is a file you can lift.

```python
"""Day 3 - structured output: schemas as data, and the validator.

The model stops writing freeform Action: lines and must hand over JSON
that matches TOOL_SCHEMAS. Bad requests die here, named, before any
tool runs. Verdicts use the same {"ok": ...} shape as the tools.
"""

import json
from typing import Any

# ----------------------------------------------------------------------------
# The schemas, as data, not code. One entry per registered tool, written
# from the real Day 1/2 signatures. Each argument maps to (type, required?).
# JSON has one number type; Python splits int/float, so a "number" field
# accepts either -- and never True/False (bool is an int subclass in Python).
# ----------------------------------------------------------------------------
TOOL_SCHEMAS: dict[str, dict[str, tuple[type, bool]]] = {
    "fetch_url":   {"url": (str, True),  "timeout": (float, False)},
    "run_command": {"command": (list, True), "timeout": (float, False)},
    "read_file":   {"rel_path": (str, True)},
}

TYPE_NAMES = {str: "string", float: "number", list: "list"}


def _type_matches(value: Any, expected: type) -> bool:
    if expected is float:   # a JSON number: int or float, never a boolean
        return isinstance(value, (int, float)) and not isinstance(value, bool)
    return isinstance(value, expected)


def _type_name(value: Any) -> str:
    """The JSON-flavored name of a value's type, for error details."""
    if isinstance(value, bool):
        return "boolean"
    if isinstance(value, str):
        return "string"
    if isinstance(value, (int, float)):
        return "number"
    if isinstance(value, list):
        return "list"
    if isinstance(value, dict):
        return "object"
    return type(value).__name__


def validate_tool_call(raw_text: str) -> dict[str, Any]:
    """Check a raw model reply against TOOL_SCHEMAS.

    Returns {"ok": True, "tool": ..., "arguments": {...}} when the call
    may run, or {"ok": False, "error": ..., "detail": ...} when it may
    not. The detail names the exact field that failed: it is written to
    be pasted into the prompt for the retry, not just logged.
    """
    # Gate 1: is it JSON at all? Prose-wrapped JSON fails here on purpose.
    try:
        call = json.loads(raw_text)
    except (json.JSONDecodeError, TypeError):
        return {"ok": False, "error": "invalid_json",
                "detail": "reply was not valid JSON"}

    # Gate 2: is it an object carrying the two keys a call must have?
    if not isinstance(call, dict) or "tool" not in call or "arguments" not in call:
        return {"ok": False, "error": "invalid_json",
                "detail": "reply must be a JSON object with 'tool' and 'arguments'"}

    name, args = call["tool"], call["arguments"]

    # Gate 3: is the tool real? If not, say what is.
    if name not in TOOL_SCHEMAS:
        return {"ok": False, "error": "unknown_tool",
                "detail": f"unknown tool '{name}'. Available: {', '.join(TOOL_SCHEMAS)}"}
    if not isinstance(args, dict):
        return {"ok": False, "error": "bad_arguments",
                "detail": "'arguments' must be an object"}

    # Gate 4: field by field: required arguments present, types matching.
    for arg, (expected, required) in TOOL_SCHEMAS[name].items():
        if arg not in args:
            if required:
                return {"ok": False, "error": "bad_arguments",
                        "detail": f"arguments.{arg} is required"}
            continue
        if not _type_matches(args[arg], expected):
            return {"ok": False, "error": "bad_arguments",
                    "detail": f"arguments.{arg} must be a {TYPE_NAMES[expected]}, "
                              f"got a {_type_name(args[arg])}"}

    return {"ok": True, "tool": name, "arguments": args}


def error_observation(verdict: dict[str, Any]) -> str:
    """Turn a failed verdict into the Observation the model will read.
    Same mechanism as Week 1's format correction; the message is sharper
    because the validator knows exactly what failed."""
    return (f"Observation: invalid tool call ({verdict['error']}): {verdict['detail']}. "
            f"Reply with ONLY a JSON object: {{\"tool\": ..., \"arguments\": {{...}}}}.")
```

### Reference: the upgraded loop (Day 4)

**Walkthrough.** Week 1's `run()`, with the parse step replaced by the validate step and dispatch by arguments. Read it against the Week 1 reference and the sameness is the point: the prompt is built once, each turn asks the model, checks for the final answer, validates or corrects, dispatches, and appends reply + Observation. Three Week 2 fingerprints: the contract text demands JSON only; a failed verdict goes back through `error_observation` and the turn repeats *without* running anything; and the final answer is itself validated (an `answer` string plus a `citations` list) before the loop will return it. The tools are called straight from the registry with `**arguments`, which is the Day 4 change to `tools.py` from the spot-the-difference exercise: `run_tool` grows from passing one string to unpacking the validated arguments.

```python
"""Day 4 - the research assistant: Week 1's loop, Week 2's contract."""

import json

from ask_gateway import ask_gateway          # your real model call (needs your key)
from structured import validate_tool_call, error_observation
from tools import TOOLS                      # your real registry: name -> function

MAX_ITERATIONS = 10   # seatbelt, same as Week 1: a confused model cannot spin forever

# The contract. Shorter than Week 1's on purpose: JSON has less to explain,
# and the worked example Week 1 needed is replaced by the schema itself.
PROMPT = """You are a research agent. You answer questions by using tools, then cite your sources.
Reply with ONLY a JSON object, no other text. To use a tool:
{"tool": "<tool name>", "arguments": { ... }}
Available tools: run_command (arguments: command, a list), read_file (arguments: rel_path),
fetch_url (arguments: url, timeout).
When you have enough to answer, reply with ONLY:
{"answer": "<your answer>", "citations": ["<file or URL>", ...]}

Question: {question}
"""


def observation_from(result):
    """Wrap a tool's return dictionary as the Observation text.
    Your tools report ok/error as data, so an error is just an
    Observation the model can react to, exactly like Week 1."""
    if result["ok"]:
        return f"Observation: {result.get('text') or result.get('stdout') or ''}"
    return f"Observation: tool error ({result['error']}): {result.get('detail', '')}"


def run(question, model=ask_gateway):
    """Ask, validate, dispatch, append, repeat. Returns the final
    answer dictionary: {"answer": str, "citations": [str, ...]}."""
    # .replace, not .format: the template's JSON braces are literal text,
    # and .format would try to read them as placeholders and crash.
    prompt = PROMPT.replace("{question}", question)

    for turn in range(1, MAX_ITERATIONS + 1):
        reply = model(prompt)

        # Final answer attempt? JSON with an "answer" key. Validate its
        # shape before accepting: an answer with no citations is a claim
        # with no receipts, and this build does not ship those.
        try:
            parsed = json.loads(reply)
        except json.JSONDecodeError:
            parsed = None
        if isinstance(parsed, dict) and "answer" in parsed:
            if isinstance(parsed["answer"], str) and isinstance(parsed.get("citations"), list):
                return parsed
            obs = ("Observation: invalid final answer: it must be a JSON object "
                   "with an 'answer' string and a 'citations' list.")

        # Otherwise it must be a tool call. VALIDATE BEFORE ANYTHING RUNS.
        else:
            verdict = validate_tool_call(reply)
            if not verdict["ok"]:
                # Rejected: the error becomes the Observation, the turn
                # repeats, and no tool ever saw the bad request.
                obs = error_observation(verdict)
            else:
                # Dispatch: the validated name picks the function, the
                # validated arguments unpack into its parameters.
                fn = TOOLS[verdict["tool"]]
                obs = observation_from(fn(**verdict["arguments"]))

        # Append reply + Observation: the growing prompt is still the
        # agent's only memory, exactly as in Week 1.
        prompt += "\n" + reply + "\n" + obs + "\n"

    return {"answer": "max iterations reached", "citations": []}
```

## Prove It: Mock vs Reality

The notebook proved the validation logic against a script. Now prove it against the real tools and a real model, where the failures are not scripted.

**1. Run your real build.** Your Day 4 research assistant, on a question you do not know the answer to, requiring all three tools. Save the trace. Verify every citation by hand: open the file, open the page. A citation you did not check is a rumor.

**2. Replay the breakages live.** The Day 5 set: a URL that 404s mid-run; a command that runs past its timeout; a question or prompt tweak that makes the model fail validation *twice in a row*. Record what the agent did each time. It must recover from at least one without dying: retry, route around, or report cleanly.

**3. Write the comparison below.** For each case: where did reality disagree with the mock? The mock's model corrected itself in one try because its script said so. The real model has read your error messages by then, or hasn't. Every disagreement is a finding. This section doubles as the week's `BREAKAGE_LOG.md` content: fill it in Friday, in your own words.

**4. Rehearse the explain test.** Out loud, no notes: why did schema validation change this agent's reliability? Which class of failures did it eliminate that Week 1's regex parsing could not catch? (Your answer should reach the word "semantic": the regex caught broken *shape*; the schema also catches right shape, wrong meaning: a real tool name with a wrong-typed argument, a string where a number belongs.)

The mock tells you the validator is correct. Reality tells you the errors are specific enough for a real model to fix itself. You need both before the week is done.

## Breakage Log: Mock vs Reality

_Fill this in Friday, in your own words. This becomes the week's `BREAKAGE_LOG.md`._

### Case 1: a tool fails mid-run (404 or timeout)
- **Mock did:**
- **Reality did:**
- **Finding:**

### Case 2: validation fails twice in a row
- **Mock did:**
- **Reality did:**
- **Finding:**

### Case 3: a citation that did not check out
- **Claim:**
- **What the source actually said:**
- **Finding:**

### Explain test rehearsal (out loud, no notes)
- [ ] a normal validated turn
- [ ] a rejected call and the retry
- [ ] a tool failure becoming an Observation
- [ ] why schemas beat regexes (the semantic failures)
- **Fuzzy spots to re-read:**

### Traces
_Paste the saved trace files from today's live runs here, or note their paths in the repo. Week 6's eval harness will eat these, and the schemas from `structured.py` are its ground truth._

## What You Own Now

- Real tools need real error contracts. The network and the filesystem do not care about your happy path, so your tools report failure as data and the loop treats it as information.
- A schema turns "hope the output parses" into "prove the output is valid." The check happens while the request is still just words, before the hands move.
- The retry is not the model getting smarter. It is the feedback getting sharper: named field, expected type, what arrived.
- Citations are not decoration. They are how you verify an agent that touched the real world, and an answer whose citations you have not checked is not done.

Your Week 2 build was the exercise for this lesson: two real tools became three, the prose parser became a schema and a validator, and the loop became a research assistant that shows its receipts. If you can narrate a validated turn, a rejection and retry, a tool failure, and a citation check out loud right now, the notebook did its job.

**Where this goes:** the ever-growing prompt you first watched in Week 1 gets worse with real tool output, and Week 3 exists to solve exactly that. And keep `structured.py` clean: in Week 6, those schemas become the ground truth your eval harness grades against.